# Histórico — avaliação externa do FP-Growth com autoria

Etapa de análise do baseline com autoria. Verifica a reprodução do membership do treino antes de usar os rótulos de origem para calcular incidência Fake/Real. Foi arquivada em `history/` porque a comparação principal está no notebook de controles sem autoria. Este notebook continua executável isoladamente.

## 1. Ambiente e implementação

As funções de leitura do corpus estão incorporadas abaixo; o notebook não requer scripts Python auxiliares. Execute as células em ordem.

In [ ]:
from pathlib import Path
import sys
NOTEBOOK_DIR = next((p / "machine-learning" / "unsupervised-learning" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "machine-learning" / "unsupervised-learning" / "README.md").is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Pasta unsupervised-learning não encontrada")
sys.path.insert(0, str(NOTEBOOK_DIR)) if str(NOTEBOOK_DIR) not in sys.path else None
print("Código:", NOTEBOOK_DIR)

### Leitura do corpus compartilhada

Mesmas features do baseline, calculadas nos primeiros 300 caracteres.

In [ ]:
"""Mineração exploratória de padrões de estilo, sem usar rótulos de classe."""

import argparse
import hashlib
import json
import re
import unicodedata
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth


ROOT = next((p / "machine-learning" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "machine-learning" / "unsupervised-learning" / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Execute a partir de uma pasta do repositório")
DATA = ROOT / "unsupervised-learning" / "data"
OUTPUTS = ROOT / "outputs" / "model-comparison"
BASELINE = OUTPUTS / "dbscan-20260924T141332Z" / "run_manifest.json"
WORDS = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
TOKENS = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)
FEATURES = ["typeTokenRatio", "linkDensity", "punctuationDensity", "uppercaseRatio", "diversidade"]
FAMILIES = {"tem_autor": "metadata", "typeTokenRatio": "lexical", "diversidade": "lexical",
            "linkDensity": "estrutura", "punctuationDensity": "estilo", "uppercaseRatio": "estilo"}

In [ ]:
def style(text, author):
    """Mesmas seis definições de estilo do notebook HDBSCAN, primeiros 300 caracteres."""
    text = unicodedata.normalize("NFKC", text).lstrip("\ufeff")[:300]
    words, tokens = WORDS.findall(text), TOKENS.findall(text)
    nw, nt = len(words), len(tokens)
    distinct = len({word.casefold() for word in words})
    links = len(re.findall(r"https?://\S+", text, flags=re.IGNORECASE))
    uppercase = sum(word.isupper() and len(word) > 1 for word in words)
    ratio = lambda a, b: a / b if b else np.nan
    return {
        "tem_autor": int(str(author).strip().casefold() not in {"", "none", "null", "nan"}),
        "typeTokenRatio": ratio(distinct, nt),
        "linkDensity": ratio(links, nw),
        "punctuationDensity": ratio(nt - nw, nt),
        "uppercaseRatio": ratio(uppercase, nw),
        "diversidade": ratio(distinct, nw),
    }

In [ ]:
def read_corpus(archive_path):
    rows = []
    with ZipFile(archive_path) as archive:
        names = archive.namelist()
        for folder in ("fake", "true"):
            marker = f"/full_texts/{folder}/"
            texts = {Path(name).stem: name for name in names if marker in name and name.endswith(".txt")}
            meta_marker = f"/full_texts/{folder}-meta-information/"
            meta = {Path(name).name.removesuffix("-meta.txt"): name for name in names
                    if meta_marker in name and name.endswith("-meta.txt")}
            if set(texts) != set(meta):
                raise ValueError("Textos e metadados não correspondem")
            for stem in sorted(texts):
                author = archive.read(meta[stem]).decode("utf-8").splitlines()[0]
                values = style(archive.read(texts[stem]).decode("utf-8"), author)
                rows.append({"record_id": f"{folder}/{stem}", **values})
    return pd.DataFrame(rows).set_index("record_id")

In [ ]:
"""Avaliação externa de padrões FP-Growth congelados; não executa mineração."""

import argparse
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact



### `load_discovery`

Carrega padrões e critérios persistidos da descoberta.

In [ ]:
def load_discovery(path):
    manifest = json.loads((path / "run_manifest.json").read_text(encoding="utf-8"))
    patterns = pd.read_csv(path / "consolidated_patterns.csv", dtype={"pattern_id": str, "display_id": str})
    criteria = pd.read_csv(path / "discretization.csv").fillna({"criterion": ""})
    membership = pd.read_csv(path / "pattern_membership.csv", dtype=str)
    patterns["features"] = patterns.features.map(json.loads)
    patterns["families"] = patterns.families.map(json.loads)
    if patterns.display_id.duplicated().any() or patterns.pattern_id.duplicated().any():
        raise ValueError("IDs de padrões duplicados")
    if set(membership.pattern_id) - set(patterns.pattern_id):
        raise ValueError("Membership contém padrão ausente da visão consolidada")
    return manifest, patterns, criteria, membership

### `apply_criteria`

Aplica literalmente os limites já congelados.

In [ ]:
def apply_criteria(features, criteria):
    """Aplica literalmente os operadores e limites persistidos na descoberta."""
    result = pd.DataFrame(index=features.index)
    for row in criteria.itertuples():
        if bool(row.omitted):
            continue
        if row.criterion == "author_present == 1":
            matched = features.tem_autor.eq(1)
        elif row.criterion == "author_present == 0":
            matched = features.tem_autor.eq(0)
        elif row.criterion == "value <= low_threshold":
            matched = features[row.feature].le(row.low_threshold)
        elif row.criterion == "value >= high_threshold":
            matched = features[row.feature].ge(row.high_threshold)
        else:
            raise ValueError(f"Critério de discretização desconhecido: {row.criterion}")
        result[row.item] = matched.fillna(False).astype(bool)
    return result

### `evaluate`

Valida o membership do treino antes de ler rótulos e calcular estatísticas.

In [ ]:
def evaluate(discovery, output):
    manifest, patterns, criteria, saved_membership = load_discovery(discovery)
    baseline = json.loads((ROOT / manifest["baseline_manifest"]).read_text(encoding="utf-8"))
    archive = DATA / f"Fake.br-Corpus-{baseline['corpus']['revision']}.zip"
    archive_sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if archive_sha != manifest["corpus_sha256"]:
        raise ValueError("Corpus difere do SHA-256 da descoberta")
    features = read_corpus(archive)
    items = apply_criteria(features, criteria)
    rows = []
    for pattern in patterns.itertuples():
        if not set(pattern.features).issubset(items.columns):
            raise ValueError(f"Item sem critério persistido: {pattern.display_id}")
        matched = items[pattern.features].all(axis=1)
        rows.extend((record_id, pattern.pattern_id, pattern.display_id)
                    for record_id in items.index[matched])
    computed = pd.DataFrame(rows, columns=["record_id", "pattern_id", "display_id"])
    # A única recomputação é para aplicar thresholds congelados fora do treino.
    train_ids = set(baseline["partitions"][manifest["protocol"]]["train"]["record_ids"])
    expected = set(map(tuple, saved_membership[["record_id", "pattern_id"]].itertuples(index=False, name=None)))
    observed = set(map(tuple, computed.loc[computed.record_id.isin(train_ids),
                                          ["record_id", "pattern_id"]].itertuples(index=False, name=None)))
    if expected != observed:
        raise ValueError(f"Membership do treino diverge da descoberta: {len(expected ^ observed)} pares")
    # Rótulos entram somente depois da validação do membership.
    labels = pd.Series(np.where(features.index.str.startswith("fake/"), "fake", "real"), index=features.index)
    if not features.index.str.match(r"^(fake|true)/").all():
        raise ValueError("record_id com origem desconhecida")
    n_fake, n_real = int(labels.eq("fake").sum()), int(labels.eq("real").sum())
    n_total = len(labels)
    baseline_fake, baseline_real = n_fake / n_total, n_real / n_total
    by_pattern = computed.groupby("pattern_id").record_id.agg(set).to_dict()
    report = []
    for pattern in patterns.itertuples():
        ids = by_pattern.get(pattern.pattern_id, set())
        fake_count = sum(labels.loc[record_id] == "fake" for record_id in ids)
        real_count = len(ids) - fake_count
        fake_pct = fake_count / len(ids) if ids else np.nan
        real_pct = real_count / len(ids) if ids else np.nan
        _, p_value = fisher_exact([[fake_count, real_count], [n_fake - fake_count, n_real - real_count]])
        report.append({"display_id": pattern.display_id, "pattern_id": pattern.pattern_id,
                       "features": pattern.features, "families": pattern.families,
                       "cross_family": pattern.cross_family, "is_primary": pattern.is_primary,
                       "representative_id": pattern.representative_id,
                       "occurrences": len(ids), "support": len(ids) / n_total,
                       "fake_count": fake_count, "real_count": real_count,
                       "fake_pct": fake_pct, "real_pct": real_pct,
                       "baseline_fake_pct": baseline_fake, "baseline_real_pct": baseline_real,
                       "delta_fake": fake_pct - baseline_fake, "delta_real": real_pct - baseline_real,
                       "lift_fake": fake_pct / baseline_fake, "lift_real": real_pct / baseline_real,
                       "discovery_support": pattern.support,
                       "discovery_max_rule_lift": pattern.max_rule_lift,
                       "discovery_max_rule_confidence": pattern.max_rule_confidence,
                       "discovery_max_rule_jaccard": pattern.max_rule_jaccard,
                       "fisher_p": p_value})
    table = pd.DataFrame(report)
    # Benjamini–Hochberg para os padrões avaliados; não altera padrões nem seleção.
    order = np.argsort(table.fisher_p.to_numpy())
    ranked = table.fisher_p.to_numpy()[order] * len(table) / np.arange(1, len(table) + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1].clip(0, 1)
    table["fisher_q_bh"] = np.nan
    table.loc[table.index[order], "fisher_q_bh"] = adjusted
    table = table.sort_values("display_id")
    per_news = computed.groupby("record_id").display_id.apply(lambda values: sorted(values)).to_dict()
    news = pd.DataFrame({"record_id": features.index,
                         "label": labels.loc[features.index].to_numpy(),
                         "matched_patterns": [per_news.get(record_id, []) for record_id in features.index]})
    news["pattern_count"] = news.matched_patterns.map(len)
    output.mkdir(parents=True, exist_ok=False)
    table.assign(features=table.features.map(json.dumps), families=table.families.map(json.dumps)).to_csv(
        output / "pattern_evaluation.csv", index=False, encoding="utf-8")
    news.assign(matched_patterns=news.matched_patterns.map(json.dumps)).to_csv(
        output / "news_patterns.csv", index=False, encoding="utf-8")
    computed.sort_values(["record_id", "display_id"]).to_csv(output / "pattern_membership.csv", index=False)
    summary = ["# Avaliação externa dos padrões FP-Growth", "",
               f"Descoberta congelada: `{discovery.name}`; notícias: {n_total}; fake: {n_fake}; real: {n_real}.",
               "", "Lift de classe = proporção da classe entre notícias com o padrão / proporção global da classe.",
               "Fisher compara presença/ausência do padrão e classe; q usa Benjamini–Hochberg. Associações são descritivas.", ""]
    for title, subset in (("Maior lift fake", table.sort_values("lift_fake", ascending=False)),
                          ("Maior lift real", table.sort_values("lift_real", ascending=False)),
                          ("Mais próximos da baseline", table.assign(distance=table.delta_fake.abs()).sort_values("distance"))):
        summary += [f"## {title}", "", "| Padrão | Features | Famílias | Ocorrências | Support | Fake % | Real % | Lift fake | Lift real | q BH |",
                    "|---|---|---|---:|---:|---:|---:|---:|---:|---:|"]
        for row in subset.head(10).itertuples():
            summary.append(f"| {row.display_id} | {{{', '.join(row.features)}}} | {' + '.join(row.families)} | "
                           f"{row.occurrences} | {row.support:.4f} | {row.fake_pct:.4f} | {row.real_pct:.4f} | "
                           f"{row.lift_fake:.4f} | {row.lift_real:.4f} | {row.fisher_q_bh:.4g} |")
        summary.append("")
    (output / "summary.md").write_text("\n".join(summary), encoding="utf-8")
    result_manifest = {"created_utc": datetime.now(timezone.utc).isoformat(),
                       "discovery_run": discovery.name, "discovery_manifest_sha256": hashlib.sha256(
                           (discovery / "run_manifest.json").read_bytes()).hexdigest(),
                       "corpus_sha256": archive_sha, "membership_train_verified": True,
                       "records": n_total, "fake": n_fake, "real": n_real, "patterns": len(table),
                       "baseline_fake": baseline_fake, "baseline_real": baseline_real,
                       "labels_used_only_after_membership": True,
                       "statistical_test": "two-sided Fisher exact; Benjamini-Hochberg across all evaluated patterns"}
    (output / "run_manifest.json").write_text(json.dumps(result_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return len(table), n_total

### `main`

Orquestra a execução e salva CSVs, resumo e manifesto.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--discovery-run", type=Path, default=OUTPUTS / "fp-growth-20260929T212058Z")
    args = parser.parse_args()
    discovery = args.discovery_run.resolve()
    if not discovery.is_dir():
        raise FileNotFoundError(discovery)
    output = OUTPUTS / f"fp-growth-evaluation-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    patterns, records = evaluate(discovery, output)
    print(f"{output}: {patterns} padrões, {records} notícias")

## 2. Protocolo e amostra

A descoberta padrão é o run histórico `fp-growth-20260929T212058Z`; substitua `DISCOVERY_RUN` por outro diretório completo se quiser avaliar um run novo. A amostra fixa de cinco padrões permite inspecionar itens, suporte e família antes da avaliação. Os limiares não são recalculados com os rótulos.

In [ ]:
DISCOVERY_RUN = OUTPUTS / "fp-growth-20260929T212058Z"
assert DISCOVERY_RUN.is_dir(), "Execute fp_growth.ipynb e aponte DISCOVERY_RUN para o novo run"
discovery_manifest, pattern_preview, criteria_preview, membership_preview = load_discovery(DISCOVERY_RUN)
print("Padrões:", len(pattern_preview), "Pares notícia/padrão no treino:", len(membership_preview))
display(pattern_preview[["display_id", "features", "families", "support"]].sample(n=min(5, len(pattern_preview)), random_state=42))

## 3. Execução

Esta célula executa o fluxo completo e grava um novo run em `machine-learning/outputs/model-comparison/`. Ajuste os parâmetros na lista abaixo se necessário. Não altere os padrões com base nos rótulos da avaliação.

In [ ]:
RUN_ARGS = ["--discovery-run", str(DISCOVERY_RUN)]
_argv = sys.argv
try:
    sys.argv = ["avaliacao_fp_growth_com_autoria.ipynb", *RUN_ARGS]
    main()
finally:
    sys.argv = _argv
run_path = max(OUTPUTS.glob("fp-growth-evaluation-*"), key=lambda p: p.stat().st_mtime if (p / "run_manifest.json").exists() else -1)
RESULT_TABLE = "pattern_evaluation.csv"
evaluation_manifest = json.loads((run_path / "run_manifest.json").read_text(encoding="utf-8"))
assert evaluation_manifest["membership_train_verified"]
assert evaluation_manifest["patterns"] == len(pattern_preview)
print("Notícias avaliadas:", evaluation_manifest["records"])

## 4. Leitura dos resultados

`fake_pct` e `real_pct` são proporções dentro de cada padrão; `lift_fake` e `lift_real` dividem essa proporção pela prevalência global da classe. Fisher é bicaudal e `fisher_q_bh` corrige os testes dos padrões avaliados. São associações descritivas, não um classificador nem validação de veracidade. Veja `docs/modelos/fp-growth-evaluation.md`.

In [ ]:
from IPython.display import display
print("Run:", run_path)
print("Arquivos:", ", ".join(sorted(p.name for p in run_path.iterdir())))
display(pd.read_csv(run_path / RESULT_TABLE).head(10))